# پروژه‌ی اول بوت‌کمپ هوش مصنوعی کوئرا — بخش سوم، مسئله‌ی ۲ (ب): پیش‌بینی قیمت فروش

**تیم:** امین رفیع · مهدی طراح · میرامیررضا سیدنصیری · سعید حیاتی · مدل فروش: میرامیررضا سیدنصیری

<div dir="rtl">

**خلاصه‌ی مسیر**

1. فقط آگهی‌های دارای `price_value` (فروش) نگه داشته شدند و ۵٪ پایین و بالای توزیع لگاریتم قیمت (قیمت‌های صفر و اشتباه تایپی) حذف شد.
2. داده **قبل از هر پیش‌پردازشی** به Train / Validation / Test (۶۴ / ۱۶ / ۲۰ درصد) تقسیم شد.
3. هر ستون خالی با یک Transformer جدا پر شد که آمارهایش **فقط روی Train** fit می‌شود. ترتیب جایگزینی: میانه یا مد گروه `cat3_slug`، بعد `cat2_slug`، بعد شهر و در آخر مقدار کلی.
   تعداد اتاق و امکانات بر اساس نوع ملک **و** سطح متراژ (کم / متوسط / زیاد) پر می‌شوند.
4. ویژگی‌های جدید: نسبت زیربنا به زمین، فضای آزاد، متراژ هر اتاق، **فاصله تا مرکز شهر** (هاورسین)، تعداد امکانات، سن بنا و تعامل متراژ و سن.
5. شهر و دسته‌ها با **Target Encoding** (با smoothing) کد شدند، چون one-hot برای ۴۰۰ شهر ابعاد را بی‌دلیل بالا می‌برد.
6. مدل **XGBoost** روی `log1p(price)` آموزش دید و Validation فقط برای early stopping استفاده شد.

**نتیجه‌ی اجرای اصلی:** R² روی Validation **0.608**، روی Test **0.606** و روی Train 0.623؛ MAE روی Test حدود **۱٫۳۶ میلیارد تومان**.
نزدیک بودن R² سه مجموعه یعنی مدل بیش‌برازش ندارد.

سلول‌ها خروجی ذخیره‌شده‌ی اجرای اصلی را دارند. سلول‌های بخش آخر (MSE و نمودارها) تازه اضافه شده‌اند و با یک بار اجرا خروجی می‌گیرند.

</div>

## ۱. بارگذاری داده

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

DIVAR_PATH = Path("../data/Divar.csv")
data_frame = pd.read_csv(DIVAR_PATH, low_memory=False)

data_frame.head(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,...,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,...,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,residential-rent,apartment-rent,ahvaz,mellirah,2024-09-01 00:00:00,NaN,اپارتمان ۱۰۰متری واحد جلو دارای دو اتاق خواب ک...,اپارتمان ۱۰۰متری ملی راه واحد جلو,مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,commercial-rent,office-rent,kermanshah,NaN,2024-11-01 00:00:00,NaN,♦مشاورین املاک سیب♦\n✅️کد:G۱۰۵\n♦با سالها سابق...,واحداداری ۸۰متری ۲۲بهمن(برق),مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,residential-sell,apartment-sell,tehran,dardasht,2024-09-01 00:00:00,مشاور املاک,♡♡♡♡♡♡بنام خدا♡♡♡♡♡♡\n♡♡♡عرض ادب واحترام♡♡♡\n♡...,اپارتمان ۱۰۰متری طبقه۳و۴ فول دردشت شمالی,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.729832,51.505466,500.0
8,8,residential-sell,apartment-sell,mahdasht-city,NaN,2024-06-01 00:00:00,مشاور املاک,با سلام\n\nاملاک بزرگ اتحاد با ۲شعبه فعال با ف...,۷۸متری دوبحرنورگیر بازسازی شده شیک سندتک برگ,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.712364,50.794781,500.0
9,9,residential-sell,apartment-sell,mashhad,faramarzabbasi,2024-10-01 00:00:00,NaN,اول رسالت فرد نزدیک به حاشیه فرامرز .. ساکنین...,متراژ ۸۰ متر فرامرز نزدیک حاشیه,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## ۲. انتخاب آگهی‌های فروش، حذف پرت‌ها و انتخاب ویژگی‌ها

**We make a copy of the main data frame.**

In [ ]:
cp_data_frame = data_frame.copy()
cp_data_frame["is_rebuilt"].isna().sum()

np.int64(470530)

 **First step is to remove the rows in the price_value column.**

In [ ]:

cp_data_frame = cp_data_frame[cp_data_frame["price_value"].notna()].copy()

price_frame = np.log1p(cp_data_frame["price_value"].astype("float64"))
lower_bound, upper_bound = price_frame.quantile(0.05), price_frame.quantile(0.95)
mask = ((price_frame >= lower_bound) & (price_frame <= upper_bound)).to_numpy()
cp_data_frame = cp_data_frame[mask].copy()


cp_data_frame.head()

,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,...,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
4,4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,residential-sell,apartment-sell,tehran,dardasht,2024-09-01 00:00:00,مشاور املاک,♡♡♡♡♡♡بنام خدا♡♡♡♡♡♡\n♡♡♡عرض ادب واحترام♡♡♡\n♡...,اپارتمان ۱۰۰متری طبقه۳و۴ فول دردشت شمالی,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.729832,51.505466,500.0
8,8,residential-sell,apartment-sell,mahdasht-city,NaN,2024-06-01 00:00:00,مشاور املاک,با سلام\n\nاملاک بزرگ اتحاد با ۲شعبه فعال با ف...,۷۸متری دوبحرنورگیر بازسازی شده شیک سندتک برگ,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.712364,50.794781,500.0
9,9,residential-sell,apartment-sell,mashhad,faramarzabbasi,2024-10-01 00:00:00,NaN,اول رسالت فرد نزدیک به حاشیه فرامرز .. ساکنین...,متراژ ۸۰ متر فرامرز نزدیک حاشیه,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**As we know the number of the lines with the Nan city_slug is less than 10 so we remove them.**


In [ ]:
cp_data_frame = cp_data_frame.dropna(subset=["city_slug"])
cp_data_frame["city_slug"].isna()

,city_slug
1,False
4,False
7,False
8,False
9,False
...,...
999990,False
999991,False
999992,False
999995,False


**In this part we just get the effective features.**

In [ ]:
cp_data_frame = cp_data_frame[["city_slug", "price_value", "land_size", "building_size", "rooms_count", "has_warehouse", "has_parking", "location_latitude"
  , "location_longitude", "cat2_slug", "cat3_slug", "construction_year","has_elevator", "has_balcony"
  ]]



## ۳. تقسیم داده (پیش از هر fit)

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(cp_data_frame, test_size=0.2, random_state=42)
train_data, validation_data = train_test_split(train_data, test_size=0.2, random_state=42)

## ۴. پر کردن مقادیر خالی (fit فقط روی Train)

**As you can see the informations in the rooms_count are in the Persian language so we map the data into English.**

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def room_count_persian(data_frame : pd.DataFrame) -> pd.DataFrame:
  map_rooms_count = {
    "دو" : 2,
    "یک" : 1,
    "سه" : 3,
    "بدون اتاق" : 0,
    "چهار" : 4,
    "پنج یا بیشتر" : 5
  }
  data_frame["rooms_count"] = data_frame["rooms_count"].map(map_rooms_count)
  return data_frame

room_count_transformer = FunctionTransformer(room_count_persian)
train_data = room_count_transformer.transform(train_data)
test_data = room_count_transformer.transform(test_data)
validation_data = room_count_transformer.transform(validation_data)


train_data.head()


,city_slug,price_value,land_size,building_size,rooms_count,has_warehouse,has_parking,location_latitude,location_longitude,cat2_slug,cat3_slug,construction_year,has_elevator,has_balcony
672026,ilam,6.500000e+09,NaN,250.0,3.0,True,True,33.641438,46.432758,residential-sell,apartment-sell,۱۴۰۳,True,NaN
84439,mashhad,8.000000e+09,326.0,190.0,2.0,True,True,NaN,NaN,residential-sell,house-villa-sell,۱۳۷۵,NaN,true
428932,firuzabad,8.500000e+08,NaN,95.0,2.0,True,True,28.831141,52.594612,residential-sell,apartment-sell,۱۳۹۴,True,NaN
783486,ahvaz,1.780000e+09,NaN,75.0,2.0,True,True,31.295769,48.718685,residential-sell,apartment-sell,۱۳۹۲,True,NaN
704347,tehran,7.200000e+09,NaN,102.0,2.0,True,True,NaN,NaN,residential-sell,apartment-sell,۱۴۰۳,True,NaN


**We group the data base on the cat3_slug , cat3_slug and city_slug. Then we fill the Nan values respectivly. This logic has been used for the rest. In some cases we used median or mode of the each group.**

*   1- > cat3_slug
*   2-> cat2_slug
* 3-> city_slug



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

class fill_constrution(BaseEstimator, TransformerMixin):
    def __init__(self, target_col: str = "construction_year", downcast: bool = True):
        self.target_col = target_col
        self.downcast = downcast
        self.cat3_medians_ = {}
        self.cat2_medians_ = {}
        self.city_medians_ = {}
        self.global_median_ = None

    def fit(self, X: pd.DataFrame, y=None):
        self.global_median_ = X[self.target_col].median()
        self.city_medians_ = X.groupby("city_slug")[self.target_col].median().to_dict()
        self.cat2_medians_ = X.groupby("cat2_slug")[self.target_col].median().to_dict()
        self.cat3_medians_ = X.groupby("cat3_slug")[self.target_col].median().to_dict()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()

        imputed_cat3 = df["cat3_slug"].map(self.cat3_medians_)
        imputed_cat2 = df["cat2_slug"].map(self.cat2_medians_)
        imputed_city = df["city_slug"].map(self.city_medians_)

        df[self.target_col] = (
            df[self.target_col]
            .fillna(imputed_cat3)
            .fillna(imputed_cat2)
            .fillna(imputed_city)
            .fillna(self.global_median_)
        )

        if self.downcast:
            df[self.target_col] = df[self.target_col].round().astype("int16")

        return df

def clean_fa_digits(series: pd.Series) -> pd.Series:
    fa_digits = "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩"
    en_digits = "01234567890123456789"
    trans_table = str.maketrans(fa_digits, en_digits)
    return series.astype(str).str.translate(trans_table)


def reshape_construction_year(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    cleaned_col = clean_fa_digits(data["construction_year"])
    cleaned_col = cleaned_col.replace(
        {
            "قبل از 1370": "1369",
            "قبل از ۱۳۷۰": "1369",
        }
    )

    data["construction_year"] = pd.to_numeric(cleaned_col, errors="coerce")
    return data


reshaper = FunctionTransformer(reshape_construction_year)
train_data = reshaper.transform(train_data)
test_data = reshaper.transform(test_data)
validation_data = reshaper.transform(validation_data)


imputer = fill_constrution(target_col="construction_year", downcast=True)
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


train_data["construction_year"]


,construction_year
672026,1403
84439,1375
428932,1394
783486,1392
704347,1403
...,...
595384,1403
819340,1369
196098,1396
934181,1396


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_building_size(BaseEstimator, TransformerMixin):
    def __init__(self, group_col: str = "city_slug", target_col: str = "building_size", downcast: bool = True):
        self.group_col = group_col
        self.target_col = target_col
        self.downcast = downcast
        self.group_medians_3 = {}
        self.group_medians_2 = {}
        self.group_medians_1 = {}
        self.global_median_ = None

    def fit(self, data_frame: pd.DataFrame, y=None):
        # calculating global median.
        self.global_median_ = data_frame[self.target_col].median()

        # calculating the median of each group based in the city_slug.
        self.group_medians_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .median()
            .to_dict()
        )
        return self

    def transform(self, data_frame, y=None):
        data_frame = data_frame.copy()
        missing_mask = data_frame[self.target_col].isna()

        if missing_mask.sum() == 0:
            return data_frame

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_medians_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_medians_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_medians_1)

        imputed_value = (
            group_imputed_3
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_median_)
        )

        data_frame.loc[missing_mask, self.target_col] = imputed_value[missing_mask]
        return data_frame

building_imputer = fill_building_size(group_col="city_slug", target_col="building_size", downcast=True)

train_data = building_imputer.fit_transform(train_data)
test_data = building_imputer.transform(test_data)
validation_data = building_imputer.transform(validation_data)


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_land_size(BaseEstimator, TransformerMixin):
    def __init__(self, group_col: str = "city_slug", target_col: str = "land_size", downcast: bool = True):
        self.group_col = group_col
        self.target_col = target_col
        self.downcast = downcast
        self.group_medians_3 = {}
        self.group_medians_2 = {}
        self.group_medians_1 = {}
        self.global_median_ = None

    def fit(self, data_frame: pd.DataFrame, y=None):
        # global median
        self.global_median_ = data_frame[self.target_col].median()

        # grouped median based on the city_slug.
        self.group_medians_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .median()
            .to_dict()
        )
        return self

    def transform(self, data_frame, y=None):
        data_frame = data_frame.copy()
        missing_mask = data_frame[self.target_col].isna()

        if missing_mask.sum() == 0:
            return data_frame

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_medians_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_medians_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_medians_1)

        imputed_value = (
            group_imputed_3
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_median_)
        )

        data_frame.loc[missing_mask, self.target_col] = imputed_value[missing_mask]
        return data_frame

imputer = fill_land_size(group_col="city_slug", target_col="land_size", downcast=True)

train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


**This cell is used to label the rows based on the building size**

In [ ]:
def building_size_patintion(data_frame : pd.DataFrame):
    df = data_frame.copy()

    df.loc[df["building_size"] < 150, "building_size_level"] = "Low"
    df.loc[(df["building_size"] >= 150) & (df["building_size"] < 300), "building_size_level"] = "Medium"
    df.loc[df["building_size"] >= 300, "building_size_level"] = "High"

    return df

building_size_patintion = FunctionTransformer(building_size_patintion)
train_data = building_size_patintion.transform(train_data)
test_data = building_size_patintion.transform(test_data)
validation_data = building_size_patintion.transform(validation_data)


train_data.columns

Index(['city_slug', 'price_value', 'land_size', 'building_size', 'rooms_count',
       'has_warehouse', 'has_parking', 'location_latitude',
       'location_longitude', 'cat2_slug', 'cat3_slug', 'construction_year',
       'has_elevator', 'has_balcony', 'building_size_level'],
      dtype='object')

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_room_count(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "rooms_count", dtype: str = "int8"):
        self.target_col = target_col
        self.dtype = dtype
        self.group_mode_3 = {}
        self.group_mode_2 = {}
        self.group_mode_1 = {}
        self.global_median_ = None

    def _calc_mode(
        self, df: pd.DataFrame, group_cols: list[str]
    ) -> dict[tuple, float]:
        mode_series = (
            df.groupby(group_cols)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .dropna()
        )
        return mode_series.to_dict()

    def fit(self, data_frame: pd.DataFrame, y=None):
        valid_mask = data_frame[self.target_col].notna()
        df_valid = data_frame.loc[valid_mask]

        self.group_mode_3 = self._calc_mode(
            df_valid, ["cat3_slug", "building_size_level"]
        )

        self.group_mode_2 = self._calc_mode(
            df_valid, ["cat2_slug", "building_size_level"]
        )

        self.group_mode_1 = self._calc_mode(
            df_valid, ["city_slug", "building_size_level"]
        )

        self.global_median_ = df_valid[self.target_col].median()

        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        keys_3 = list(
            zip(data_frame["cat3_slug"], data_frame["building_size_level"])
        )
        mode_imputed_3 = pd.Series(keys_3, index=data_frame.index).map(
            self.group_mode_3
        )
        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            mode_imputed_3
        )

        if data_frame[self.target_col].isna().any():
            keys_2 = list(
                zip(data_frame["cat2_slug"], data_frame["building_size_level"])
            )
            mode_imputed_2 = pd.Series(keys_2, index=data_frame.index).map(
                self.group_mode_2
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_2
            )

        if data_frame[self.target_col].isna().any():
            keys_1 = list(
                zip(data_frame["city_slug"], data_frame["building_size_level"])
            )
            mode_imputed_1 = pd.Series(keys_1, index=data_frame.index).map(
                self.group_mode_1
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_1
            )

        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            self.global_median_
        )

        data_frame[self.target_col] = np.round(
            data_frame[self.target_col]
        ).astype(self.dtype)

        return data_frame

imputer = fill_room_count()
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)



**The normalize_bool_col is used to turn booleans to inteegers**
1.   True : 1
2.   False : 0



In [ ]:
def normalize_bool_col(series: pd.Series) -> pd.Series:
    mapping = {
        True: 1, False: 0,
        'True': 1, 'False': 0,
        'true': 1, 'false': 0,
        '1': 1, '0': 0,
        1: 1, 0: 0,
    }
    return series.map(mapping)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_elavator(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_elevator", dtype: str = "int8"):
        self.target_col = target_col
        self.dtype = dtype
        self.group_mode_3 = {}
        self.group_mode_2 = {}
        self.group_mode_1 = {}
        self.global_median_ = None

    def _calc_mode(
        self, df: pd.DataFrame, group_cols: list[str]
    ) -> dict[tuple, float]:
        mode_series = (
            df.groupby(group_cols)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .dropna()
        )
        return mode_series.to_dict()

    def fit(self, data_frame: pd.DataFrame, y=None):
        valid_mask = data_frame[self.target_col].notna()
        df_valid = data_frame.loc[valid_mask]

        self.group_mode_3 = self._calc_mode(
            df_valid, ["cat3_slug", "building_size_level"]
        )

        self.group_mode_2 = self._calc_mode(
            df_valid, ["cat2_slug", "building_size_level"]
        )

        self.group_mode_1 = self._calc_mode(
            df_valid, ["city_slug", "building_size_level"]
        )

        self.global_median_ = df_valid[self.target_col].median()

        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        keys_3 = list(
            zip(data_frame["cat3_slug"], data_frame["building_size_level"])
        )
        mode_imputed_3 = pd.Series(keys_3, index=data_frame.index).map(
            self.group_mode_3
        )
        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            mode_imputed_3
        )

        if data_frame[self.target_col].isna().any():
            keys_2 = list(
                zip(data_frame["cat2_slug"], data_frame["building_size_level"])
            )
            mode_imputed_2 = pd.Series(keys_2, index=data_frame.index).map(
                self.group_mode_2
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_2
            )

        if data_frame[self.target_col].isna().any():
            keys_1 = list(
                zip(data_frame["city_slug"], data_frame["building_size_level"])
            )
            mode_imputed_1 = pd.Series(keys_1, index=data_frame.index).map(
                self.group_mode_1
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_1
            )

        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            self.global_median_
        )

        data_frame[self.target_col] = np.round(
            data_frame[self.target_col]
        ).astype(self.dtype)

        return data_frame

train_data['has_elevator'] = normalize_bool_col(train_data['has_elevator'])
test_data['has_elevator'] = normalize_bool_col(test_data['has_elevator'])
validation_data['has_elevator'] = normalize_bool_col(validation_data['has_elevator'])



imputer = fill_elavator()
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class fill_balcany(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_balcony", dtype: str = "int8"):
        self.target_col = target_col
        self.dtype = dtype
        self.group_mode_3 = {}
        self.group_mode_2 = {}
        self.group_mode_1 = {}
        self.global_median_ = None

    def _calc_mode(
        self, df: pd.DataFrame, group_cols: list[str]
    ) -> dict[tuple, float]:
        mode_series = (
            df.groupby(group_cols)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .dropna()
        )
        return mode_series.to_dict()

    def fit(self, data_frame: pd.DataFrame, y=None):
        df = data_frame.copy()
        df[self.target_col] = normalize_bool_col(df[self.target_col])

        target_series = df[self.target_col]

        valid_mask = target_series.notna()
        df_valid = df.loc[valid_mask].copy()

        self.group_mode_3 = self._calc_mode(
            df_valid, ["cat3_slug", "building_size_level"]
        )
        self.group_mode_2 = self._calc_mode(
            df_valid, ["cat2_slug", "building_size_level"]
        )
        self.group_mode_1 = self._calc_mode(
            df_valid, ["city_slug", "building_size_level"]
        )

        self.global_median_ = df_valid[self.target_col].median()

        return self


    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        data_frame[self.target_col] = pd.to_numeric(
            data_frame[self.target_col], errors="coerce"
        )

        index_3 = pd.MultiIndex.from_arrays(
            [data_frame["cat3_slug"], data_frame["building_size_level"]]
        )
        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            pd.Series(index_3.map(self.group_mode_3), index=data_frame.index)
        )

        if data_frame[self.target_col].isna().any():
            index_2 = pd.MultiIndex.from_arrays(
                [data_frame["cat2_slug"], data_frame["building_size_level"]]
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                pd.Series(
                    index_2.map(self.group_mode_2), index=data_frame.index
                )
            )

        if data_frame[self.target_col].isna().any():
            index_1 = pd.MultiIndex.from_arrays(
                [data_frame["city_slug"], data_frame["building_size_level"]]
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                pd.Series(
                    index_1.map(self.group_mode_1), index=data_frame.index
                )
            )

        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            self.global_median_
        )

        data_frame[self.target_col] = np.round(
            data_frame[self.target_col]
        ).astype(self.dtype)

        return data_frame

imputer = fill_balcany()
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


In this part we want to fill the has_parking.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


train_data["has_parking"] = normalize_bool_col(train_data["has_parking"])
test_data["has_parking"] = normalize_bool_col(test_data["has_parking"])
validation_data["has_parking"] = normalize_bool_col(validation_data["has_parking"])


class fill_has_parking(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_parking"):
        self.target_col = target_col
        self.group_modes_3 = {}
        self.group_modes_2 = {}
        self.group_modes_1 = {}
        self.global_mode_ = None

    @staticmethod
    def _calc_mode(series: pd.Series):
        m = series.mode()
        return m.iloc[0] if not m.empty else np.nan

    def fit(self, data_frame: pd.DataFrame, y=None):

        m = data_frame[self.target_col].mode()
        self.global_mode_ = m.iloc[0] if not m.empty else np.nan

        self.group_modes_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_modes_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_modes_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_modes_1)

        data_frame[self.target_col] = (
            data_frame[self.target_col]
            .fillna(group_imputed_3)
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_mode_)
        )

        return data_frame



imputer = fill_has_parking(target_col="has_parking")

train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


print(train_data["has_parking"].value_counts(dropna=False))


has_parking
1.0    280016
0.0     47584
Name: count, dtype: int64


In [ ]:
class fill_cordinate(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        target_col: str,
        city_col: str = "city_slug",
        dtype: str = "float64"
    ):
        self.target_col = target_col
        self.city_col = city_col
        self.dtype = dtype

        self.city_median_ = {}
        self.global_median_ = None

    def fit(self, data_frame: pd.DataFrame, y=None):
        valid_mask = data_frame[self.target_col].notna()
        df_valid = data_frame.loc[valid_mask]

        # calculating the medain based on the city.
        self.city_median_ = (
            df_valid.groupby(self.city_col)[self.target_col]
            .median()
            .dropna()
            .to_dict()
        )

        # global median.
        self.global_median_ = df_valid[self.target_col].median()

        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        # based on the city.
        imputed_l2 = data_frame[self.city_col].map(self.city_median_)
        data_frame[self.target_col] = data_frame[self.target_col].fillna(imputed_l2)

        # global median.
        data_frame[self.target_col] = data_frame[self.target_col].fillna(self.global_median_)

        # Casting
        data_frame[self.target_col] = data_frame[self.target_col].astype(self.dtype)

        return data_frame


lat_imputer = fill_cordinate(
    target_col="location_latitude",
    city_col="city_slug",
    dtype="float64"
)

lon_imputer = fill_cordinate(
    target_col="location_longitude",
    city_col="city_slug",
    dtype="float64"
)

train_data = lat_imputer.fit_transform(train_data)
train_data = lon_imputer.fit_transform(train_data)

validation_data = lat_imputer.transform(validation_data)
validation_data = lon_imputer.transform(validation_data)

test_data = lat_imputer.transform(test_data)
test_data = lon_imputer.transform(test_data)



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

train_data['has_warehouse'] = normalize_bool_col(train_data['has_warehouse'])
test_data['has_warehouse'] = normalize_bool_col(test_data['has_warehouse'])


class fill_has_warehouse(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_warehouse"):
        self.target_col = target_col
        self.group_modes_3 = {}
        self.group_modes_2 = {}
        self.group_modes_1 = {}
        self.global_mode_ = None

    @staticmethod
    def _calc_mode(series: pd.Series):
        m = series.mode()
        return m.iloc[0] if not m.empty else np.nan

    def fit(self, data_frame: pd.DataFrame, y=None):

        m = data_frame[self.target_col].mode()
        self.global_mode_ = m.iloc[0] if not m.empty else np.nan

        self.group_modes_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_modes_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_modes_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_modes_1)

        data_frame[self.target_col] = (
            data_frame[self.target_col]
            .fillna(group_imputed_3)
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_mode_)
        )

        return data_frame


imputer = fill_has_warehouse(target_col="has_warehouse")

train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


print(train_data["has_warehouse"].value_counts(dropna=False))


has_warehouse
1.0    289010
0.0     38590
Name: count, dtype: int64


## ۵. کدگذاری و مهندسی ویژگی

**In this part label the data based on the building_level which wa created before.
Also we want to the categorical feature in our data frame.**

In [ ]:
cat_cols = ["city_slug", "cat2_slug", "cat3_slug"]

categories = {
    c: pd.Index(train_data[c].dropna().unique())
    for c in cat_cols
}

for c in cat_cols:
    train_data[c] = pd.Categorical(train_data[c], categories=categories[c])
    test_data[c]  = pd.Categorical(test_data[c],  categories=categories[c])
    validation_data[c] = pd.Categorical(validation_data[c], categories=categories[c])

level_map = {"Low": 1, "Medium": 2, "High": 3}
train_data["building_size_level"] = (
    train_data["building_size_level"].map(level_map).astype("UInt8")
)
test_data["building_size_level"] = (
    test_data["building_size_level"].map(level_map).astype("UInt8")
)
validation_data["building_size_level"] = (
    validation_data["building_size_level"].map(level_map).astype("UInt8")
)


train_data = train_data.dropna(subset=["price_value"])
test_data = test_data.dropna(subset=["price_value"])

train_x = train_data.drop(columns=["price_value"])
test_x = test_data.drop(columns=["price_value"])
val_x = validation_data.drop(columns=["price_value"])

train_y = train_data["price_value"]
test_y = test_data["price_value"]
val_y = validation_data["price_value"]


In [ ]:
city_lat_center = train_x.groupby('city_slug')['location_latitude'].median()
city_lon_center = train_x.groupby('city_slug')['location_longitude'].median()


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_features(df, city_lat_center, city_lon_center):
    df = df.copy()

    df['building_land_ratio'] = df['building_size'] / (df['land_size'] + 1)
    df["free_area"] = df["land_size"]  - df["building_size"]
    df['area_per_room'] = df['building_size'] / df['rooms_count'].replace(0, 1)

    center_lat = df['city_slug'].map(city_lat_center).astype(float)
    center_lon = df['city_slug'].map(city_lon_center).astype(float)
    lat = df['location_latitude'].astype(float).values
    lon = df['location_longitude'].astype(float).values
    df['dist_to_city_center'] = haversine_distance(
        df['location_latitude'].values,
        df['location_longitude'].values,
        center_lat.values,
        center_lon.values,
    )
    df['amenity_count'] = (
    df['has_warehouse'].astype(int) +
    df['has_parking'].astype(int) +
    df['has_elevator'].astype(int) +
    df['has_balcony'].astype(int)
    )

    df['building_age'] = (1403 - df['construction_year']).clip(0, 100)

    df['size_age_interact'] = df['building_size'] / (df['building_age'] + 1)

    label = (df["building_age"] <= 10)
    df[label]["is_newly_built"] = 1
    df[~label]["is_newly_built"] = 0

    new_cols = [
        'building_land_ratio',
        'area_per_room',
        'dist_to_city_center',
        'building_age',
        'size_age_interact',
        "free_area",
        "amenity_count"
    ]
    for col in new_cols:
        df[col] = df[col].fillna(df[col].median())

    return df

train_x = add_features(train_x, city_lat_center, city_lon_center)
val_x = add_features(val_x, city_lat_center, city_lon_center)
test_x = add_features(test_x, city_lat_center, city_lon_center)



**Give a code to each category in the data frame bases on its weigth to the target.**

In [ ]:
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['city_slug','cat2_slug','cat3_slug'], smoothing=10)
train_x = encoder.fit_transform(train_x, train_y)
val_x   = encoder.transform(val_x)
test_x   = encoder.transform(test_x)

In [ ]:
val_x['has_parking'] = normalize_bool_col(val_x['has_parking']).fillna(0).astype('int8')
test_x['has_parking'] = normalize_bool_col(test_x['has_parking']).fillna(0).astype('int8')
train_x['has_parking'] = normalize_bool_col(train_x['has_parking']).fillna(0).astype('int8')

train_x['has_warehouse'] = normalize_bool_col(train_x['has_warehouse'])
val_x['has_warehouse']   = normalize_bool_col(val_x['has_warehouse'])
test_x['has_warehouse']  = normalize_bool_col(test_x['has_warehouse'])

object_cols = train_x.select_dtypes(include='object').columns.tolist()
if object_cols:
    print("Still object dtype:", object_cols)
else:
    print("All dtypes OK — ready for XGBoost")


All dtypes OK — ready for XGBoost


## ۶. مدل XGBoost

In this case we use a xgboost to test our preproccessing.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import numpy as np

train_y = np.log1p(train_y)
test_y  = np.log1p(test_y)
val_y   = np.log1p(val_y)

model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.1,
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=100,
    eval_metric='rmse',
    n_jobs=-1,
    enable_categorical=True,
)

model.fit(
    train_x, train_y,
    eval_set=[(val_x, val_y)],
    verbose=200
)

y_pred_log = model.predict(val_x)
y_pred = np.expm1(y_pred_log)

val_y = np.expm1(val_y)
train_y = np.expm1(train_y)
test_y = np.expm1(test_y)

print(f"MAE    : {mean_absolute_error(val_y, y_pred):,.0f}")
print(f"R2     : {r2_score(val_y, y_pred):.4f}")

train_pred = np.expm1(model.predict(train_x))
print(f"Train R2: {r2_score(train_y, train_pred):.4f}")

test_pred = np.expm1(model.predict(test_x))
print(f"MAE Test    : {mean_absolute_error(test_y, test_pred):,.0f}")
print(f"R2 Test     : {r2_score(test_y, test_pred):.4f}")


[0]	validation_0-rmse:0.88930
[200]	validation_0-rmse:0.58477
[400]	validation_0-rmse:0.56938
[600]	validation_0-rmse:0.56083
[800]	validation_0-rmse:0.55529
[1000]	validation_0-rmse:0.55072
[1200]	validation_0-rmse:0.54748
[1400]	validation_0-rmse:0.54482
[1600]	validation_0-rmse:0.54268
[1800]	validation_0-rmse:0.54089
[1999]	validation_0-rmse:0.53936
MAE    : 1,356,662,691
R2     : 0.6078
Train R2: 0.6234
MAE Test    : 1,355,622,714
R2 Test     : 0.6062


## ۷. گزارش کامل معیارها (R²، MAE، MSE)

<div dir="rtl">

صورت پروژه R²، MAE و MSE را روی داده‌ی Test می‌خواهد. سلول بالا فقط R² و MAE را چاپ می‌کرد؛ این‌جا هر سه معیار، هم به تومان و هم روی مقیاس لگاریتمی که مدل با آن آموزش دیده، گزارش می‌شوند.

</div>

In [ ]:
from sklearn.metrics import mean_squared_error


def report(y_true, y_pred, name):
    return {
        "split": name,
        "R2": r2_score(y_true, y_pred),
        "MAE (billion Toman)": mean_absolute_error(y_true, y_pred) / 1e9,
        "MSE (Toman^2)": mean_squared_error(y_true, y_pred),
        "RMSE (billion Toman)": np.sqrt(mean_squared_error(y_true, y_pred)) / 1e9,
        "R2 (log)": r2_score(np.log1p(y_true), np.log1p(y_pred)),
        "MAE (log)": mean_absolute_error(np.log1p(y_true), np.log1p(y_pred)),
        "MSE (log)": mean_squared_error(np.log1p(y_true), np.log1p(y_pred)),
        "median abs % error": np.median(np.abs(y_pred - y_true) / y_true) * 100,
    }


metrics = pd.DataFrame([report(train_y, train_pred, "train"),
                        report(val_y, y_pred, "validation"),
                        report(test_y, test_pred, "test")]).set_index("split")
metrics.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hexbin(np.log10(test_y), np.log10(test_pred), gridsize=70, bins="log", cmap="viridis")
lims = [np.log10(test_y).min(), np.log10(test_y).max()]
axes[0].plot(lims, lims, color="red", linewidth=1)
axes[0].set_xlabel("actual price (log10 Toman)")
axes[0].set_ylabel("predicted price (log10 Toman)")
axes[0].set_title("Test set: predicted vs actual")

importance = pd.Series(model.feature_importances_, index=train_x.columns).sort_values()
importance.tail(15).plot(kind="barh", ax=axes[1], color="#2471a3")
axes[1].set_title("XGBoost feature importance (gain share)")
plt.tight_layout()
plt.show()

## ۸. جمع‌بندی

<div dir="rtl">

- مدل نهایی حدود **۶۱٪ واریانس قیمت** را روی داده‌ی دیده‌نشده توضیح می‌دهد. نزدیکی R² در Train، Validation و Test نشان می‌دهد مدل بیش‌برازش نکرده است.
- قیمت فروش در این داده ترکیبی از آپارتمان، ویلا، زمین، پیش‌فروش و ملک تجاری در صدها شهر است و ویژگی‌های مهمی مثل کیفیت ساخت، دقیق بودن موقعیت و سند در آن نیست. برای همین سقف دقیقی که می‌شود انتظار داشت محدود است.
- MAE حدود ۱٫۳ میلیارد تومان در برابر میانه‌ی قیمت حدود ۳ میلیارد زیاد به نظر می‌رسد، ولی خطا نسبی است: در ملک‌های ارزان کم و در ملک‌های گران زیاد است. معیار «میانه‌ی درصد خطا» در جدول بالا تصویر واقعی‌تری می‌دهد.
- **ریسک ارزیابی:** ۱۰٪ دو سر توزیع قیمت، قبل از تقسیم و برای Test هم، حذف شده است. پس معیارها برای قیمت‌های «معمول» معتبرند و نه برای ملک‌های خیلی ارزان یا خیلی گران.
- قدم‌های بعدی: مدل جدا برای هر نوع ملک (آپارتمان، ویلا، زمین)، استفاده از محله با Target Encoding، و تنظیم پارامترها با جست‌وجوی سیستماتیک روی Validation.

</div>